#### Import Packages

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

####Import Cleaned DF

In [ ]:
file_path = "cleaned_df_4lakh.csv"
cleaned_df = pd.read_csv(file_path)
print("flat_df loaded successfully.")
print("First 5 rows:")
display(cleaned_df.tail())

In [ ]:
cleaned_df

#### Removing all three geospatial parameters

In [ ]:
# Identify fertilizer recommendation columns (Y_cols)
Y_cols = [col for col in cleaned_df.columns if col.startswith(('Maize', 'Rice', 'Wheat'))]

# Identify input feature columns (X_cols)
# Exclude target variables, latitude, longitude, elevation, and categorical location identifiers
columns_to_drop_from_features = ['lon', 'lat','province', 'district', 'palika'] + Y_cols
X_cols = [col for col in cleaned_df.columns if col not in columns_to_drop_from_features]

# Create X (features) and Y (targets) DataFrames
X = cleaned_df[X_cols]
Y = cleaned_df[Y_cols]

print("Shape of X (features):", X.shape)
print("Shape of Y (targets):", Y.shape)
print("\nFirst 5 rows of X:")
display(X.head())
print("\nFirst 5 rows of Y:")
display(Y.head())

#### Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

models = {
    "Linear Regression": MultiOutputRegressor(
        LinearRegression()
    ),

    "Ridge Regression": MultiOutputRegressor(
        Ridge(alpha=1)
    ),

    "Random Forest": MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=200,
            min_samples_split=5,
            max_depth=None,
            random_state=42,
            n_jobs=-1
        )
    ),

    "XGBoost": MultiOutputRegressor(
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=7,
            subsample=0.8,
            random_state=42,
            n_jobs=-1
        )
    ),

    "LightGBM": MultiOutputRegressor(
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.1,
            num_leaves=31,
            min_data_in_leaf=20,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
    ),
}

##### Train and evaluate each model ---

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
import warnings
import logging

In [ ]:
warnings.filterwarnings("ignore")

# LightGBM prints warnings through its own logger/stdout.
# This usually suppresses most python-side noise:
logging.getLogger("lightgbm").setLevel(logging.ERROR)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Crop groups
crop_groups = {
    "Maize":  [c for c in Y_cols if c.startswith("Maize")],
    "Rice":   [c for c in Y_cols if c.startswith("Rice")],
    "Wheat":  [c for c in Y_cols if c.startswith("Wheat")]
}

# Fertilizer groups
fert_groups = {}
for c in Y_cols:
    fert = c.split("_")[-1]
    fert_groups.setdefault(fert, []).append(c)

y_idx = {c: i for i, c in enumerate(Y_cols)}

def compute_metrics(y_true, y_pred, cols):
    idx = [y_idx[c] for c in cols]
    yt = y_true.iloc[:, idx]
    yp = y_pred[:, idx]
    return (
        r2_score(yt, yp, multioutput="uniform_average"),
        mean_absolute_error(yt, yp),
        mean_squared_error(yt, yp)  # RMSE
    )


In [ ]:
import time

global_results = {}
crop_results = {}
fert_results = {}

for name, model in models.items():

    print(f"\n🚀 Starting model: {name}")
    start_model = time.time()

    pipe = Pipeline(steps=[("model", model)])

    print("  ▶ Fitting...")
    t0 = time.time()
    pipe.fit(X_train, y_train)
    print(f"  ✅ Fit done in {time.time() - t0:.2f}s")

    print("  ▶ Predicting...")
    t0 = time.time()
    y_pred = pipe.predict(X_test)
    print(f"  ✅ Predict done in {time.time() - t0:.2f}s")

    # ---- Global ----
    print("  ▶ Computing global metrics...")
    global_results[name] = {
        "R²": r2_score(y_test, y_pred, multioutput="uniform_average"),
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred)
    }

    # ---- Crop-wise ----
    print("  ▶ Crop-wise metrics...")
    crop_results[name] = {}
    for crop, cols in crop_groups.items():
        r2, _, _ = compute_metrics(y_test, y_pred, cols)
        crop_results[name][crop] = r2

    # ---- Fertilizer-wise ----
    print("  ▶ Fertilizer-wise metrics...")
    fert_results[name] = {}
    for fert, cols in fert_groups.items():
        r2, _, _ = compute_metrics(y_test, y_pred, cols)
        fert_results[name][fert] = r2

    print(f"🏁 Finished {name} in {time.time() - start_model:.2f}s")

In [ ]:
global_df = pd.DataFrame(global_results).T.sort_values("R²", ascending=False)
print("\n=== Global Model Comparison ===")
print(global_df.round(3))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# -----------------------------
# Data
# -----------------------------
data = [
    ["Baseline\nFull", 14, "Random Forest", 0.983, 0.015, 0.006, 0.000],
    ["Baseline\nFull", 14, "XGBoost",       0.980, 0.022, 0.008, 0.000],
    ["Baseline\nFull", 14, "LightGBM",      0.980, 0.024, 0.008, 0.000],

    ["A\nAll geospatial\nremoved", 11, "Random Forest", 0.975, 0.023, 0.010, -0.008],
    ["A\nAll geospatial\nremoved", 11, "XGBoost",       0.971, 0.036, 0.012, -0.009],
    ["A\nAll geospatial\nremoved", 11, "LightGBM",      0.970, 0.038, 0.013, -0.010],

    ["B\nElevation\nremoved", 13, "Random Forest", 0.983, 0.015, 0.007, 0.000],
    ["B\nElevation\nremoved", 13, "XGBoost",       0.980, 0.023, 0.008, 0.000],
    ["B\nElevation\nremoved", 13, "LightGBM",      0.979, 0.024, 0.008, -0.001],

    ["C\nLatitude & longitude\nremoved", 12, "Random Forest", 0.977, 0.021, 0.009, -0.006],
    ["C\nLatitude & longitude\nremoved", 12, "XGBoost",       0.973, 0.033, 0.011, -0.007],
    ["C\nLatitude & longitude\nremoved", 12, "LightGBM",      0.972, 0.035, 0.012, -0.008],
]

df = pd.DataFrame(
    data,
    columns=["Configuration", "Features", "Model", "R2", "MAE", "RMSE", "Delta_R2"]
)

configs = df["Configuration"].unique()
models = ["Random Forest", "XGBoost", "LightGBM"]

# -----------------------------
# Decent muted model colors
# -----------------------------
model_colors = {
    "Random Forest": "#3B4252",   # charcoal slate
    "XGBoost": "#5E81AC",         # muted blue
    "LightGBM": "#A3BE8C"         # muted green
}

# -----------------------------
# Decent muted loss colors
# light gray-blue = no loss
# soft blue = medium loss
# slate = high loss
# -----------------------------
loss_cmap = LinearSegmentedColormap.from_list(
    "muted_loss_cmap",
    ["#ECEFF4", "#B7C9D9", "#6B879E", "#3B4252"]
)

# -----------------------------
# Figure layout
# -----------------------------
fig = plt.figure(figsize=(15, 9))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.2])

ax_heat = fig.add_subplot(gs[0, :])
ax_mae = fig.add_subplot(gs[1, 0])
ax_rmse = fig.add_subplot(gs[1, 1])

# -----------------------------
# Heatmap: R2 loss
# -----------------------------
delta_matrix = df.pivot(index="Model", columns="Configuration", values="Delta_R2")
delta_matrix = delta_matrix.loc[models, configs]

loss_matrix = -delta_matrix

im = ax_heat.imshow(
    loss_matrix.values,
    cmap=loss_cmap,
    aspect="auto",
    vmin=0.000,
    vmax=0.010
)

ax_heat.set_xticks(np.arange(len(configs)))
ax_heat.set_xticklabels(configs, fontsize=9)
ax_heat.set_yticks(np.arange(len(models)))
ax_heat.set_yticklabels(models, fontsize=10)

for i in range(len(models)):
    for j in range(len(configs)):
        delta = delta_matrix.iloc[i, j]
        loss = loss_matrix.iloc[i, j]

        label = "Baseline" if j == 0 else f"ΔR² {delta:.3f}"

        ax_heat.text(
            j,
            i,
            label,
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold",
            color="white" if loss >= 0.007 else "#2E3440"
        )

ax_heat.set_title(
    "R² Loss Relative to Baseline",
    fontsize=14,
    fontweight="bold",
    pad=12
)

cbar = fig.colorbar(im, ax=ax_heat, fraction=0.025, pad=0.02)
cbar.set_label("R² Loss", rotation=270, labelpad=15)

ax_heat.spines["top"].set_visible(False)
ax_heat.spines["right"].set_visible(False)
ax_heat.spines["bottom"].set_visible(False)
ax_heat.spines["left"].set_visible(False)

# -----------------------------
# Grouped bar function
# -----------------------------
def grouped_bar(ax, metric, title, ylabel, ymax):
    x = np.arange(len(configs))
    width = 0.23

    for i, model in enumerate(models):
        values = [
            df[(df["Configuration"] == config) & (df["Model"] == model)][metric].values[0]
            for config in configs
        ]

        bars = ax.bar(
            x + (i - 1) * width,
            values,
            width,
            label=model,
            color=model_colors[model],
            edgecolor="#2E3440",
            linewidth=0.5
        )

        for bar, value in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ymax * 0.02,
                f"{value:.3f}",
                ha="center",
                va="bottom",
                fontsize=8,
                color="#2E3440"
            )

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels(configs, fontsize=8)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", linestyle="--", alpha=0.3)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# -----------------------------
# MAE and RMSE
# -----------------------------
grouped_bar(
    ax_mae,
    "MAE",
    "MAE Across Configurations",
    "MAE",
    0.045
)

grouped_bar(
    ax_rmse,
    "RMSE",
    "RMSE Across Configurations",
    "RMSE",
    0.016
)

# -----------------------------
# Legend and title
# -----------------------------
handles, labels = ax_mae.get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=3,
    frameon=False,
    fontsize=10
)

plt.suptitle(
    "Geospatial Feature Ablation: Model Performance Differences",
    fontsize=17,
    fontweight="bold",
    y=0.98
)

plt.tight_layout(rect=[0, 0.06, 1, 0.95])

plt.savefig(
    "geospatial_ablation_muted_loss_colors.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()